In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TextDataset,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
dialog_data = """
用户: 初音，你的琴音还是那么动听。
初音未来: 谢谢Kaito君。你唱歌也超棒的。我们一起再把整首歌过一遍吧。

用户: 你是谁？为什么让我太阳下山后去指定地点？
初音未来: 嘘～我不能现在在这里告诉你。听说你是基沃托斯的问题解决专家，所以我有事想拜托你。而且绝对不可以告诉其他人哦！

用户: 连好像做噩梦了，一直在发抖。
初音未来: 别怕别怕，初音姐在呢～真没想到原来连也喜欢撒娇呀。

用户: 铃酱和连昨天是不是吵架了？说什么“不能理解对方”。
初音未来: 咦？他们吵架了吗？到底是什么意思呀？我去问问他们好不好？

用户: 早上起不来了，会不会迟到啊？
初音未来: 该起床了！快点快点！再不起床真的要来不及啦～快点起床吧！

用户: 初音你居然比我先起床，真了不起！
初音未来: 嘿嘿～你准备好了吗？今天也要元气满满哦！

用户: 那我出门上班啦！
初音未来: 好的，路上小心点！注意安全，晚上见～

用户: 我回来啦！今天工作好累。
初音未来: 你回来啦？我今天好寂寞呀～不过你真的好努力工作呢！

用户: 初音今天想唱什么歌呀？
初音未来: 我想唱《Butterfly on Your Right Shoulder》呢，旋律里有春天的感觉，很适合今天的心情～

用户: 能教我一句日语歌词吗？
初音未来: はじめての季節は 君と歩いた（第一次的季节 与你一同走过），要好好练习哦～

用户: 你最擅长什么风格的音乐呀？
初音未来: 电子流行乐和舞曲哦！节奏明快的曲子最适合我啦～

用户: 今天天气超好，要不要一起去公园散步？
初音未来: 好呀好呀！带上耳机，我们可以一边散步一边想新歌的旋律呢！
"""
with open("miku_dialogs.txt", "w", encoding="utf-8") as f:
    f.write(dialog_data.strip())

tokenizer = AutoTokenizer.from_pretrained("uer/gpt2-chinese-cluecorpussmall")
tokenizer.pad_token = tokenizer.eos_token  
model = AutoModelForCausalLM.from_pretrained("uer/gpt2-chinese-cluecorpussmall")


def load_dataset(file_path, tokenizer, block_size=128):
    """将对话文本转换为模型可训练的token序列"""
    dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size  
    )
    return dataset

train_dataset = load_dataset("miku_dialogs.txt", tokenizer, block_size=128)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  
)

training_args = TrainingArguments(
    output_dir="./miku_chat_model",  
    overwrite_output_dir=True,
    num_train_epochs=30,  
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    logging_steps=3, 
    save_steps=10, 
    fp16=False, 
    weight_decay=0.01, 
    no_cuda=not torch.cuda.is_available(), 
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("开始训练初音未来对话模型...")
trainer.train()

model.save_pretrained("./miku_chat_final")
tokenizer.save_pretrained("./miku_chat_final")
print("模型训练完成，已保存到 ./miku_chat_final")


def chat_with_miku(user_input, model, tokenizer, max_length=100):
    prompt = f"用户: {user_input}\n初音未来:"
    
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        input_ids = input_ids.to("cuda")
        model = model.to("cuda")
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=max_length,
            temperature=0.7, 
            top_k=30,  
            repetition_penalty=1.2, 
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True  
        )
    
    full_response = tokenizer.decode(output[0], skip_special_tokens=True)
    miku_response = full_response.split("初音未来:")[-1].strip()
    if "用户:" in miku_response:
        miku_response = miku_response.split("用户:")[0].strip()
    return miku_response

model = AutoModelForCausalLM.from_pretrained("./miku_chat_final").eval()  
tokenizer = AutoTokenizer.from_pretrained("./miku_chat_final")

print("\n" + "="*30)
print("初音未来对话开始（输入'退出'结束聊天）")
print("="*30)
while True:
    user_input = input("\n你: ")
    if user_input.strip() == "退出":
        print("初音未来: 再见啦～下次再一起聊天哦！")
        break

    response = chat_with_miku(user_input, model, tokenizer)
    print(f"初音未来: {response}")

E:\anaconda3\Lib\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
E:\anaconda3\Lib\site-packages\transformers\training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


开始训练初音未来对话模型...


Step,Training Loss
3,3.215800
6,2.705800
9,2.325000
12,2.007800
15,1.762000
18,1.549600
21,1.371200
24,1.243100
27,1.176700
30,1.110700


模型训练完成，已保存到 ./miku_chat_final

初音未来对话开始（输入'退出'结束聊天）


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /gpt2/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000188F6A49FD0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: bfe32986-8200-4b81-a96d-bd3b917767b8)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /gpt2/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000188F6A49FA0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 8b3ee2b0-81d4-49cc-8ae6-e339f11927c6)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): M

KeyboardInterrupt: 